# Task 1: Data Cleaning and Preprocessing

**Objective:** Clean and prepare a raw dataset — handle nulls, duplicates, and inconsistent formats.  
**Dataset:** Mall Customer Segmentation (from Kaggle)  
**Tools:** Python (Pandas, NumPy)

## Step 1 — Import Libraries

In [1]:
import pandas as pd
import numpy as np

# pandas  -> data manipulation
# numpy   -> numerical operations & injecting NaN for demo

## Step 2 — Load the Dataset

Download `Mall_Customers.csv` from Kaggle and place it in the same folder as this notebook.

In [2]:
df = pd.read_csv('Mall_Customers.csv')

# Preview the first 5 rows
df.head()

,CustomerID,Gender,Age,Annual Income (k$),Spending Score (1-100)
0,1,Male,19,15,39
1,2,Male,21,15,81
2,3,Female,20,16,6
3,4,Female,23,16,77
4,5,Female,31,17,40


## Step 3 — Explore & Understand the Data

Before cleaning, always explore the dataset to understand its structure, size, and data types.

In [3]:
# Shape: (rows, columns)
print('Shape:', df.shape)

# Column names, data types, non-null counts
df.info()

Shape: (200, 5)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 200 entries, 0 to 199
Data columns (total 5 columns):
 #   Column                  Non-Null Count  Dtype 
---  ------                  --------------  ----- 
 0   CustomerID              200 non-null    int64 
 1   Gender                  200 non-null    object
 2   Age                     200 non-null    int64 
 3   Annual Income (k$)      200 non-null    int64 
 4   Spending Score (1-100)  200 non-null    int64 
dtypes: int64(4), object(1)
memory usage: 7.9+ KB


In [4]:
# Basic statistics for numeric columns
df.describe()

,CustomerID,Age,Annual Income (k$),Spending Score (1-100)
count,200.000000,200.000000,200.000000,200.000000
mean,100.500000,38.850000,60.560000,50.200000
std,57.879185,13.969007,26.264721,25.823522
min,1.000000,18.000000,15.000000,1.000000
25%,50.750000,28.750000,41.500000,34.750000
50%,100.500000,36.000000,61.500000,50.000000
75%,150.250000,49.000000,78.000000,73.000000
max,200.000000,70.000000,137.000000,99.000000


## Step 4 — Rename Column Headers

Clean column names: lowercase + replace spaces with underscores.  
This avoids errors like `df['Annual Income']` vs `df['annual_income']`.

In [5]:
# Lowercase + strip whitespace + replace spaces with underscores
df.columns = df.columns.str.lower().str.strip().str.replace(' ', '_')

print('Cleaned column names:')
print(df.columns.tolist())

Cleaned column names:
['customerid', 'gender', 'age', 'annual_income_(k$)', 'spending_score_(1-100)']


## Step 5 — Handle Missing Values

- `isnull().sum()` counts NaN per column  
- **Numeric columns** fill with **median** (robust to outliers)  
- **Categorical columns** fill with **mode** (most frequent value)

In [6]:
# Inject artificial nulls for demonstration
df.loc[5:8, 'age'] = np.nan
df.loc[10:12, 'genre'] = np.nan

print('Missing values before:')
print(df.isnull().sum())

Missing values before:
customerid                  0
gender                      0
age                         4
annual_income_(k$)          0
spending_score_(1-100)      0
genre                     200
dtype: int64


In [7]:
# Fill numeric column with median
df['age'] = df['age'].fillna(df['age'].median())

# Fill categorical column with mode
df['gender'] = df['gender'].fillna(df['gender'].mode()[0])

print('Missing values after:')
print(df.isnull().sum())

Missing values after:
customerid                  0
gender                      0
age                         0
annual_income_(k$)          0
spending_score_(1-100)      0
genre                     200
dtype: int64


## Step 6 — Remove Duplicate Rows

`drop_duplicates()` removes rows where ALL column values are identical.  
`reset_index(drop=True)` re-numbers the index after removal.

In [8]:
# Inject a duplicate row for demonstration
df = pd.concat([df, df.iloc[[0]]], ignore_index=True)

print('Duplicates before:', df.duplicated().sum())

df = df.drop_duplicates()
df = df.reset_index(drop=True)

print('Duplicates after: ', df.duplicated().sum())

Duplicates before: 1
Duplicates after:  0


## Step 7 — Standardize Text Values

Inconsistent values like `male`, `Male`, `MALE` all mean the same thing.  
`.str.strip()` removes whitespace. `.str.title()` applies Title Case.

In [9]:
print('Unique genders before:', df['gender'].unique())

# Strip whitespace and apply title case
df['gender'] = df['gender'].str.strip().str.title()

print('Unique genders after: ', df['gender'].unique())

Unique genders before: ['Male' 'Female' 'female' 'male']
Unique genders after:  ['Male' 'Female']


## Step 8 — Fix Data Types

After `fillna()`, numeric columns may become `float64`.  
- `age` should be `int` (no decimals)  
- `customerid` is an identifier, store as `str`

In [10]:
# Convert age to integer
df['age'] = df['age'].astype(int)

# CustomerID is an identifier, keep as string
df['customerid'] = df['customerid'].astype(str)

# Verify all dtypes
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 200 entries, 0 to 199
Data columns (total 6 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   customerid              200 non-null    object 
 1   gender                  200 non-null    object 
 2   age                     200 non-null    int64  
 3   annual_income_(k$)      200 non-null    int64  
 4   spending_score_(1-100)  200 non-null    int64  
 5   genre                   0 non-null      float64
dtypes: float64(1), int64(3), object(2)
memory usage: 9.5+ KB


## Step 9 — Detect & Treat Outliers (IQR Capping)

**IQR** = Q3 - Q1  
- Lower = Q1 - 1.5 x IQR  
- Upper = Q3 + 1.5 x IQR  

Values outside boundaries are **capped** instead of dropped — keeps all rows intact.

In [11]:
def cap_outliers(df, col):
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    print(f'{col}: lower={lower:.1f}, upper={upper:.1f}')
    df[col] = df[col].clip(lower=lower, upper=upper)
    return df

for col in ['age', 'annual_income_(k$)', 'spending_score_(1-100)']:
    df = cap_outliers(df, col)

print('Outliers capped successfully.')

age: lower=-1.0, upper=79.0
annual_income_(k$): lower=-13.2, upper=132.8
spending_score_(1-100): lower=-22.6, upper=130.4
Outliers capped successfully.


## Step 10 — Final Data Quality Check

In [12]:
print('=== Final Shape ===', df.shape)
print('\n=== Missing Values ===')
print(df.isnull().sum())
print('\n=== Data Types ===')
print(df.dtypes)
print('\n=== Sample ===')
df.head()

=== Final Shape === (200, 6)

=== Missing Values ===
customerid                  0
gender                      0
age                         0
annual_income_(k$)          0
spending_score_(1-100)      0
genre                     200
dtype: int64

=== Data Types ===
customerid                 object
gender                     object
age                         int64
annual_income_(k$)        float64
spending_score_(1-100)      int64
genre                     float64
dtype: object

=== Sample ===


,customerid,gender,age,annual_income_(k$),spending_score_(1-100),genre
0,1,Male,19,15.0,39,NaN
1,2,Male,21,15.0,81,NaN
2,3,Female,20,16.0,6,NaN
3,4,Female,23,16.0,77,NaN
4,5,Female,31,17.0,40,NaN


## Step 11 — Export Cleaned Dataset

In [13]:
# index=False avoids saving row numbers as an extra column
df.to_csv('Mall_Customers_Cleaned.csv', index=False)
print('Cleaned dataset saved as Mall_Customers_Cleaned.csv')

Cleaned dataset saved as Mall_Customers_Cleaned.csv
